# Align FOV positions across microscopes

Re-image the **same FOVs** on a second microscope after physically moving the
stage insert. Two stages:

**Part 1 — boundary alignment (coarse).** Assumes no rotation and a rigid sample,
so the two stage coordinate systems differ only by an **isotropic similarity
transform** — one uniform scale plus a translation, optionally with a **per-axis
flip** (a mirror on x and/or y):

$$q_x = (\pm s)\,p_x + t_x \qquad q_y = (\pm s)\,p_y + t_y$$

The transform is found by **overlapping the two tissue-boundary polygons** (a
closed-form centroid/area match, refined by maximising polygon IoU; the four
axis-flip combinations are tried and the best overlap kept), then applied to the
source FOV positions to produce coarse target positions.

**Part 2 — bead drift refinement (optional).** After imaging fiducial beads on
the target scope at the coarse positions, measure and correct the residual
per-FOV drift via `phase_cross_correlation` (see that section below).

**Inputs** — each is an **explicit path** (its own directory; edit in the
parameter cells):
- source boundary, target boundary (comma-separated `x,y` vertices)
- source FOV positions (`positions_{SAMPLE_NAME}.txt`)
- (Part 2) per-FOV source & target bead images + their frame tables

**Outputs** (in `OUTPUT_DIR`): coarse + drift-corrected target positions and
diagnostic plots.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.io             import load_positions, save_positions_array
from MERci.acquisition.alignment import (
    load_boundary_polygon, fit_isotropic_alignment,   # Part 1: boundary alignment
    select_bead_frame, compute_fov_drifts,            # Part 2: bead drift refinement
)

SAMPLE_NAME = SAMPLE_DIR.name
print(f"SAMPLE_DIR  : {SAMPLE_DIR}")
print(f"SAMPLE_NAME : {SAMPLE_NAME}")

In [ ]:
# ── Part 1 parameters (boundary alignment) ───────────────────
# Microscope labels (used to build the default per-directory paths below).
SOURCE_LABEL = "mf4"    # where the run was already imaged (epifluorescence)
TARGET_LABEL = "mf2"    # where you want to re-image the same FOVs (confocal)

# Each boundary / positions input is an EXPLICIT path with its own directory —
# edit any of these to wherever the file actually lives. The defaults put each
# microscope's files in its own positions/<label>/ subfolder.
SOURCE_BOUNDARY  = SAMPLE_DIR / "positions" / SOURCE_LABEL / "boundary_positions.txt"
TARGET_BOUNDARY  = SAMPLE_DIR / "positions" / TARGET_LABEL / "boundary_positions.txt"
SOURCE_POSITIONS = SAMPLE_DIR / "positions" / SOURCE_LABEL / f"positions_{SAMPLE_NAME}.txt"

# Where corrected positions + diagnostic plots are written.
OUTPUT_DIR = SAMPLE_DIR / "positions" / TARGET_LABEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Coarse (boundary-only) target positions output.
TARGET_POSITIONS = OUTPUT_DIR / f"positions_{SAMPLE_NAME}_{TARGET_LABEL}.txt"

REFINE     = True   # maximise polygon IoU after the closed-form centroid/area init
ALLOW_FLIP = True   # also try x/y axis flips (mirroring) and keep the best overlap

for p in (SOURCE_BOUNDARY, TARGET_BOUNDARY, SOURCE_POSITIONS):
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

In [ ]:
# ── Load boundaries and source FOV positions ──────────────────────────
src_poly = load_boundary_polygon(SOURCE_BOUNDARY)
tgt_poly = load_boundary_polygon(TARGET_BOUNDARY)

pos_dict     = load_positions(SOURCE_POSITIONS)              # {fov_id: (x, y)}
src_fovs     = np.array([pos_dict[i] for i in sorted(pos_dict)], dtype=float)

print(f"Source boundary : {len(src_poly.exterior.coords) - 1:4d} vertices, area={src_poly.area:,.0f}")
print(f"Target boundary : {len(tgt_poly.exterior.coords) - 1:4d} vertices, area={tgt_poly.area:,.0f}")
print(f"Source FOVs     : {len(src_fovs)}")

In [ ]:
# ── Fit the transform (isotropic scale + translation + optional axis flips) ──
fit = fit_isotropic_alignment(src_poly, tgt_poly, refine=REFINE, allow_flip=ALLOW_FLIP)

print(f"scale        : {fit.scale:.5f}")
print(f"translation  : ({fit.tx:,.2f}, {fit.ty:,.2f})")
print(f"flips        : x={fit.flip_x}, y={fit.flip_y}")
print(f"IoU (init)   : {fit.iou_init:.4f}")
print(f"IoU (final)  : {fit.iou:.4f}   {'(refined)' if fit.refined else '(closed-form)'}")
if fit.flip_x or fit.flip_y:
    flipped = ", ".join(ax for ax, f in (("x", fit.flip_x), ("y", fit.flip_y)) if f)
    print(f"\nNote: the {flipped}-axis is mirrored between {SOURCE_LABEL} and {TARGET_LABEL}.")
if fit.iou < 0.8:
    print("\n⚠  Low overlap — check that the boundaries are from the same sample,")
    print("   that the sample is not rotated (only flips/scale/shift are corrected),")
    print("   and that both files are valid. Try toggling ALLOW_FLIP.")

In [ ]:
# ── Apply the transform to the FOV positions ─────────────────────────
tgt_fovs = fit.transform_points(src_fovs)

print(f"Mapped {len(tgt_fovs)} FOV positions into the {TARGET_LABEL} coordinate system.")
print(f"  source x ∈ [{src_fovs[:,0].min():.0f}, {src_fovs[:,0].max():.0f}], "
      f"y ∈ [{src_fovs[:,1].min():.0f}, {src_fovs[:,1].max():.0f}]")
print(f"  target x ∈ [{tgt_fovs[:,0].min():.0f}, {tgt_fovs[:,0].max():.0f}], "
      f"y ∈ [{tgt_fovs[:,1].min():.0f}, {tgt_fovs[:,1].max():.0f}]")

In [ ]:
# ── Visualise the alignment ─────────────────────────────────────
def _ring(poly):
    return np.asarray(poly.exterior.coords)

src_ring  = _ring(src_poly)
tgt_ring  = _ring(tgt_poly)
warp_ring = _ring(fit.transform_polygon(src_poly))

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 6))

ax0.plot(src_ring[:, 0], src_ring[:, 1], "-", color="tab:blue", label=f"{SOURCE_LABEL} boundary")
ax0.plot(tgt_ring[:, 0], tgt_ring[:, 1], "-", color="tab:orange", label=f"{TARGET_LABEL} boundary")
ax0.scatter(src_fovs[:, 0], src_fovs[:, 1], s=6, color="tab:blue", alpha=0.5, label=f"{SOURCE_LABEL} FOVs")
ax0.set_title("Before: raw coordinates")
ax0.legend(loc="best", fontsize=8)
ax0.set_aspect("equal"); ax0.grid(alpha=0.3)

ax1.plot(tgt_ring[:, 0], tgt_ring[:, 1], "-", color="tab:orange", lw=2, label=f"{TARGET_LABEL} boundary")
ax1.plot(warp_ring[:, 0], warp_ring[:, 1], "--", color="tab:blue", label=f"{SOURCE_LABEL}→{TARGET_LABEL} boundary")
ax1.scatter(tgt_fovs[:, 0], tgt_fovs[:, 1], s=6, color="tab:blue", alpha=0.6, label="mapped FOVs")
ax1.set_title(f"After: IoU = {fit.iou:.3f}")
ax1.legend(loc="best", fontsize=8)
ax1.set_aspect("equal"); ax1.grid(alpha=0.3)

flip_txt = f", flip x={fit.flip_x}/y={fit.flip_y}" if (fit.flip_x or fit.flip_y) else ""
fig.suptitle(f"{SOURCE_LABEL} → {TARGET_LABEL}  "
             f"(scale={fit.scale:.4f}, t=({fit.tx:.0f}, {fit.ty:.0f}){flip_txt})")
fig.tight_layout()

plot_path = OUTPUT_DIR / f"alignment_{SAMPLE_NAME}_{SOURCE_LABEL}_to_{TARGET_LABEL}.png"
fig.savefig(plot_path, dpi=150)
print(f"Saved diagnostic plot: {plot_path}")
plt.show()

In [ ]:
# ── Save the coarse (boundary-only) target FOV positions ─────────────
# These are the positions to image beads at on the target scope; Part 2 below
# refines them per-FOV using the bead images.
save_positions_array(tgt_fovs, TARGET_POSITIONS)
print(f"Saved {len(tgt_fovs)} coarse positions: {TARGET_POSITIONS}")

## Part 2 — Bead drift refinement (optional)

The boundary alignment above is a **coarse** whole-sample transform. After moving
the stage insert to the target microscope and imaging fiducial **beads** at the
coarse positions, this section measures and corrects the small residual per-FOV
drift.

- **Source bead images** = the source scope's DAPI/cells series (move them to a
  network location reachable from the target scope).
- **Target bead images** = beads imaged on the target scope, one stack per FOV at
  the coarse positions.
- For each FOV the bead frame (auto-detected from the frame table) is registered
  with `skimage.registration.phase_cross_correlation` — the same primitive
  [fishtank](https://github.com/jweissmanlab/fishtank)'s `align_experiments` uses
  for its coarse stage — giving one **(dx, dy) drift vector per FOV**.

**Outputs** (in `OUTPUT_DIR`): drift-corrected positions
(`positions_{SAMPLE}_{TARGET}_drift_corrected.txt`), a quiver plot of per-FOV
displacement, and a drift-distance histogram with mean ± std.

> Confirm `DRIFT_SIGN_X/Y` against the quiver plot — the image-pixel → stage-axis
> sign convention is microscope-specific. This refines **xy** drift on a single
> bead z-slice.

In [ ]:
# ── Part 2 parameters (bead drift) ───────────────────────────────────
# Source bead images = the compact bead-only files produced by
# `extract_source_bead_frames.ipynb` at the source scope and copied to the NAS
# (full-resolution bead frames only — phase cross-correlation needs the image).
# Target bead images = beads imaged on the target scope at the coarse positions.
# ONE file per FOV in each dir.
SOURCE_BEAD_DIR = SAMPLE_DIR / "beads_source"                  # <-- edit (NAS path)
TARGET_BEAD_DIR = SAMPLE_DIR / "data" / f"beads_{TARGET_LABEL}"   # <-- edit

# Per-FOV filename patterns (Python format with {fov}); match your file naming.
SOURCE_BEAD_PATTERN = f"beads_{SOURCE_LABEL}_{{fov:03d}}.tiff"     # from the extract notebook
TARGET_BEAD_PATTERN = f"hal-{TARGET_LABEL}-beads_{{fov:03d}}.dax"  # <-- edit

# .dax needs frame_width/height; .zarr / .tiff embed their own dims (leave None).
FRAME_W = None
FRAME_H = None

# Bead-frame selection (read from each series' frame table). For the compact
# source files, use the compact frame table written by extract_source_bead_frames.
SOURCE_FRAME_TABLE = SOURCE_BEAD_DIR / f"frame_table_beads_{SOURCE_LABEL}.csv"      # <-- edit
TARGET_FRAME_TABLE = SAMPLE_DIR / "metadata" / "frame_table_405f25-488f2-seq.csv"  # <-- edit
BEAD_COLOR = None      # None = auto-detect the single-z fiducial colour (e.g. 488)
BEAD_WHICH = "first"   # which fiducial frame: "first" / "last" / "middle"

# Target microscope pixel size (µm/px) for pixel→stage conversion.
TARGET_PIXEL_SIZE_UM = 0.108
UPSAMPLE = 20          # subpixel precision (1/UPSAMPLE px)

# Image-pixel → target-stage axis-sign mapping. CONFIRM against the quiver plot.
DRIFT_SIGN_X = 1.0
DRIFT_SIGN_Y = 1.0

print("Source bead dir :", SOURCE_BEAD_DIR)
print("Target bead dir :", TARGET_BEAD_DIR)

In [ ]:
# ── Measure per-FOV bead drift ───────────────────────────────────────
src_ft = pd.read_csv(SOURCE_FRAME_TABLE, index_col=0)
tgt_ft = pd.read_csv(TARGET_FRAME_TABLE, index_col=0)
ref_frame = select_bead_frame(src_ft, bead_color=BEAD_COLOR, which=BEAD_WHICH)
mov_frame = select_bead_frame(tgt_ft, bead_color=BEAD_COLOR, which=BEAD_WHICH)
print(f"Bead frame — source: {ref_frame} (color {src_ft.loc[ref_frame, 'color']}), "
      f"target: {mov_frame} (color {tgt_ft.loc[mov_frame, 'color']})")

fov_ids = sorted(pos_dict)   # same order as the src_fovs / tgt_fovs rows
pairs = []
for fov in fov_ids:
    sp = SOURCE_BEAD_DIR / SOURCE_BEAD_PATTERN.format(fov=fov)
    tp = TARGET_BEAD_DIR / TARGET_BEAD_PATTERN.format(fov=fov)
    if sp.exists() and tp.exists():
        pairs.append((fov, sp, tp))
print(f"{len(pairs)}/{len(fov_ids)} FOVs have both source and target bead images.")

drift = compute_fov_drifts(
    pairs, ref_frame=ref_frame, mov_frame=mov_frame,
    pixel_size_um=TARGET_PIXEL_SIZE_UM, upsample_factor=UPSAMPLE,
    sign_x=DRIFT_SIGN_X, sign_y=DRIFT_SIGN_Y,
    frame_width=FRAME_W, frame_height=FRAME_H,
)
print(drift[["dx_px", "dy_px", "drift_x_um", "drift_y_um"]].describe().loc[["mean", "std", "min", "max"]])
drift.head()

In [ ]:
# ── Apply per-FOV drift to the coarse positions ──────────────────────
coarse_by_fov = {fov: tgt_fovs[i] for i, fov in enumerate(fov_ids)}
drift_by_fov  = {int(r.fov): np.array([r.drift_x_um, r.drift_y_um])
                 for _, r in drift.iterrows()}

coarse_arr    = np.array([coarse_by_fov[f] for f in fov_ids])
corrected_arr = np.array([coarse_by_fov[f] + drift_by_fov.get(f, np.zeros(2))
                          for f in fov_ids])

n_corr = sum(1 for f in fov_ids if f in drift_by_fov)
print(f"Applied drift to {n_corr}/{len(fov_ids)} FOVs "
      f"({len(fov_ids) - n_corr} kept at the coarse position).")

CORRECTED_POSITIONS = OUTPUT_DIR / f"positions_{SAMPLE_NAME}_{TARGET_LABEL}_drift_corrected.txt"
save_positions_array(corrected_arr, CORRECTED_POSITIONS)
print(f"Saved drift-corrected positions: {CORRECTED_POSITIONS}")

In [ ]:
# ── Per-FOV drift vectors (coarse → corrected) ───────────────────────
# Drifts are µm-scale while the FOV layout spans mm, so arrows are magnified for
# visibility. Use this plot to CONFIRM the DRIFT_SIGN_X/Y convention: each arrow
# should point from the coarse position toward where the beads actually landed.
ARROW_MAG = 50

measured = [f for f in fov_ids if f in drift_by_fov]
cf = np.array([coarse_by_fov[f] for f in measured]) if measured else np.empty((0, 2))
dd = np.array([drift_by_fov[f]  for f in measured]) if measured else np.empty((0, 2))

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(coarse_arr[:, 0], coarse_arr[:, 1], s=10, color="tab:blue",
           alpha=0.5, label="coarse (boundary) FOVs")
if len(cf):
    ax.quiver(cf[:, 0], cf[:, 1], dd[:, 0] * ARROW_MAG, dd[:, 1] * ARROW_MAG,
              angles="xy", scale_units="xy", scale=1, color="tab:red",
              width=0.003, label=f"drift × {ARROW_MAG}")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
ax.set_title(f"Per-FOV bead drift ({SOURCE_LABEL}→{TARGET_LABEL})")
fig.tight_layout()
quiver_path = OUTPUT_DIR / f"drift_vectors_{SAMPLE_NAME}_{TARGET_LABEL}.png"
fig.savefig(quiver_path, dpi=150)
print(f"Saved drift vectors: {quiver_path}")
plt.show()

In [ ]:
# ── Drift-distance distribution (mean ± std) ─────────────────────────
dist_um = np.hypot(drift["drift_x_um"].to_numpy(), drift["drift_y_um"].to_numpy())
m, s = float(dist_um.mean()), float(dist_um.std())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(dist_um, bins=20, color="tab:gray", edgecolor="k", alpha=0.85)
ax.axvline(m, color="tab:red", lw=2, label=f"mean = {m:.3f} µm")
ax.axvspan(m - s, m + s, color="tab:red", alpha=0.15, label=f"±std = {s:.3f} µm")
ax.set_xlabel("drift distance (µm)")
ax.set_ylabel("FOV count")
ax.set_title(f"Per-FOV bead drift distribution ({len(dist_um)} FOVs)")
ax.legend()
fig.tight_layout()
hist_path = OUTPUT_DIR / f"drift_distribution_{SAMPLE_NAME}_{TARGET_LABEL}.png"
fig.savefig(hist_path, dpi=150)
print(f"Saved drift histogram: {hist_path}")
plt.show()